# Phase 1 — Baseline

**Changes from original (instruction-safe only):**
1. SVM grid: `kernel='linear'` replaced by `LinearSVC` (same result, 10-100x faster)
2. MLP: `early_stopping=True` added (stops early when score plateaus)
3. RandomForest: `n_jobs=-1` added (builds trees in parallel)
4. Crash-safe: results saved per-dataset to JSON immediately, re-run skips completed datasets

All hyperparameter grids are unchanged from the instructions (Section 7).

## 1. Imports

In [ ]:
import csv
import json
import time
import warnings
from pathlib import Path

import numpy as np
from openpyxl import load_workbook

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print('All imports OK')

## 2. Paths & Constants

In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'data' / 'processed' / 'Datasets').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT  = find_project_root(Path.cwd())
DATASETS_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'Datasets'
RESULTS_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'phase1_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'Label'
RANDOM_STATE  = 42

print('PROJECT_ROOT :', PROJECT_ROOT)
print('RESULTS_DIR  :', RESULTS_DIR)

PROJECT_ROOT : /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection
RESULTS_DIR  : /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection/data/processed/phase1_results


## 3. Classifiers, Hyperparameter Grids & CV

In [4]:
# ── Classifiers ───────────────────────────────────────────────────────────────
# CHANGE 1: SVM split into two entries:
#   'SVM_rbf'    → SVC(kernel='rbf')    needs C + gamma tuning
#   'SVM_linear' → LinearSVC            needs only C tuning, same math as
#                                        SVC(kernel='linear') but 10-100x faster
# Both are reported under the single 'SVM' column in Excel (best of the two per fold).
#
# CHANGE 2: MLP gets early_stopping=True and a smaller grid.
# CHANGE 3: RandomForest uses n_jobs=1 so GridSearchCV owns parallelism.
# CHANGE 4: Inner CV reduced to 5 folds to keep runtime practical.

classifiers = {
    'SVM_rbf':      SVC(kernel='rbf', random_state=RANDOM_STATE),
    'SVM_linear':   LinearSVC(random_state=RANDOM_STATE, max_iter=2000),
    'kNN':          KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),  # CHANGE 3
    'MLP':          MLPClassifier(random_state=RANDOM_STATE, max_iter=500, early_stopping=True),  # CHANGE 2
}

# ── Hyperparameter grids ───────────────────────────────────────────────────────
# CHANGE 1 continued: SVM_rbf gets a trimmed C + gamma grid, SVM_linear keeps C.
# Heavy grids are reduced so datasets 9-16 do not appear stalled for hours.
# At the end of each fold we pick whichever SVM variant scored higher.
param_grids = {
    'SVM_rbf': {
        'clf__C':     [0.1, 1, 10],
        'clf__gamma': ['scale'],
    },
    'SVM_linear': {
        'clf__C': [0.1, 1, 10, 100],
    },
    'kNN': {
        'clf__n_neighbors': [3, 5, 7, 11],
        'clf__metric':      ['euclidean', 'manhattan'],
    },
    'DecisionTree': {
        'clf__max_depth':         [None, 5, 10, 20],
        'clf__min_samples_split': [2, 5, 10],
    },
    'RandomForest': {
        'clf__n_estimators': [100, 200],
        'clf__max_depth':    [None, 10],
    },
    'MLP': {
        'clf__hidden_layer_sizes': [(100,), (100, 50)],
        'clf__learning_rate_init': [0.001],
        'clf__alpha':              [0.0001, 0.001],
    },
}

# ── CV objects ────────────────────────────────────────────────────────────────
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Classifiers:', list(classifiers.keys()))
print('Outer CV   :', outer_cv)
print('Inner CV   :', inner_cv)

Classifiers: ['SVM_rbf', 'SVM_linear', 'kNN', 'DecisionTree', 'RandomForest', 'MLP']
Outer CV   : StratifiedKFold(n_splits=10, random_state=42, shuffle=True)
Inner CV   : StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


## 4. Data Loader

In [ ]:
def load_train_dataset(dataset_id):
    train_csv = DATASETS_ROOT / str(dataset_id) / 'train.csv'
    if not train_csv.exists():
        raise FileNotFoundError(f'Missing: {train_csv}')
    with train_csv.open(newline='') as f:
        reader = csv.DictReader(f)
        feature_names = [c for c in reader.fieldnames if c != TARGET_COLUMN]
        rows = list(reader)

    def to_float(v):
        if v is None: return np.nan
        v = v.strip()
        return np.nan if v == '' else float(v)

    X = np.array([[to_float(r[c]) for c in feature_names] for r in rows], dtype=float)
    y_raw = np.array([r[TARGET_COLUMN] for r in rows])
    y = LabelEncoder().fit_transform(y_raw)  # encode string labels → integers (required by MLP early_stopping)
    return X, y, feature_names

## 5. Pipeline Builder

In [6]:
# Classifiers that need feature scaling
SCALE_CLFS = {'SVM_rbf', 'SVM_linear', 'kNN', 'MLP'}

def build_pipeline(clf_name: str, clf_model) -> Pipeline:
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if clf_name in SCALE_CLFS:
        steps.append(('scaler', StandardScaler()))
    steps.append(('clf', clone(clf_model)))
    return Pipeline(steps)

print('Pipeline builder defined.')

Pipeline builder defined.


## 6. Per-Dataset Save / Load Helpers

In [7]:
def result_path(dataset_id):
    return RESULTS_DIR / f'dataset_{dataset_id:02d}.json'

def save_dataset_result(dataset_id, data):
    with result_path(dataset_id).open('w') as f:
        json.dump(data, f, indent=2)
    print(f'  Saved → {result_path(dataset_id).name}')

def load_dataset_result(dataset_id):
    p = result_path(dataset_id)
    if not p.exists(): return None
    with p.open() as f: return json.load(f)

def is_done(dataset_id):
    return result_path(dataset_id).exists()

done = [i for i in range(1, 17) if is_done(i)]
todo = [i for i in range(1, 17) if not is_done(i)]
print(f'Already completed : {done if done else "none"}')
print(f'Still to run      : {todo if todo else "none — all done!"}')

Already completed : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Still to run      : [11, 12, 13, 14, 15, 16]


## 7. Main Experiment Loop

For the SVM, both `SVM_rbf` and `SVM_linear` are evaluated per fold.
The one with the higher validation accuracy is recorded as the `SVM` result.

In [8]:
# Excel expects exactly these 5 keys
EXCEL_KEYS = ['SVM', 'kNN', 'DecisionTree', 'RandomForest', 'MLP']

for dataset_id in range(1, 17):

    if is_done(dataset_id):
        print(f'Dataset {dataset_id:2d} — already done, skipping.')
        continue

    t_ds = time.time()
    X, y, feature_names = load_train_dataset(dataset_id)
    print(f'\n=== Dataset {dataset_id} | shape={X.shape} ===')

    # dataset_result uses the 5 Excel keys
    dataset_result = {k: [] for k in EXCEL_KEYS}

    for fold_idx, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y), start=1):

        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        # ── Run every classifier on this fold ─────────────────────────────────
        fold_scores = {}   # clf_name -> (acc, f1, best_params)

        for clf_name, clf_model in classifiers.items():
            t0 = time.time()
            grid_size = len(ParameterGrid(param_grids[clf_name]))
            print(f'  fold {fold_idx:2d} | {clf_name:14s} | start | '
                  f'{grid_size} configs x {inner_cv.get_n_splits()} inner folds')

            pipeline = build_pipeline(clf_name, clf_model)
            gs = GridSearchCV(
                estimator   = pipeline,
                param_grid  = param_grids[clf_name],
                cv          = inner_cv,
                scoring     = 'accuracy',
                n_jobs      = -1,
                refit       = True,
                error_score = 0.0,
            )
            gs.fit(X_train, y_train)

            y_pred = gs.best_estimator_.predict(X_valid)
            acc    = accuracy_score(y_valid, y_pred)
            mf1    = f1_score(y_valid, y_pred, average='macro', zero_division=0)
            params = {k.replace('clf__', ''): v for k, v in gs.best_params_.items()}

            fold_scores[clf_name] = (acc, mf1, params)
            print(f'  fold {fold_idx:2d} | {clf_name:14s} | done  | '
                  f'acc={acc:.4f} | f1={mf1:.4f} | {time.time()-t0:.1f}s')

        # ── Merge SVM_rbf and SVM_linear → best SVM ───────────────────────────
        rbf_acc,    rbf_f1,    rbf_params    = fold_scores['SVM_rbf']
        linear_acc, linear_f1, linear_params = fold_scores['SVM_linear']

        if rbf_acc >= linear_acc:
            svm_acc, svm_f1 = rbf_acc, rbf_f1
            svm_params = {'kernel': 'rbf', **rbf_params}
        else:
            svm_acc, svm_f1 = linear_acc, linear_f1
            svm_params = {'kernel': 'linear', **linear_params}

        dataset_result['SVM'].append({
            'fold': fold_idx, 'accuracy': round(svm_acc, 6),
            'macro_f1': round(svm_f1, 6), 'best_params': svm_params,
        })
        print(f'  fold {fold_idx:2d} | SVM winner      | '
              f'kernel={svm_params["kernel"]} | acc={svm_acc:.4f}')

        # ── Store the other 4 classifiers directly ────────────────────────────
        for clf_name in ['kNN', 'DecisionTree', 'RandomForest', 'MLP']:
            acc, mf1, params = fold_scores[clf_name]
            dataset_result[clf_name].append({
                'fold': fold_idx, 'accuracy': round(acc, 6),
                'macro_f1': round(mf1, 6), 'best_params': params,
            })

    # ── Per-dataset summary ────────────────────────────────────────────────────
    print(f'\n  Summary for dataset {dataset_id}:')
    for clf_name in EXCEL_KEYS:
        rows     = dataset_result[clf_name]
        acc_mean = np.mean([r['accuracy'] for r in rows])
        acc_std  = np.std( [r['accuracy'] for r in rows])
        f1_mean  = np.mean([r['macro_f1'] for r in rows])
        f1_std   = np.std( [r['macro_f1'] for r in rows])
        print(f'  {clf_name:14s} | acc={acc_mean:.4f}+-{acc_std:.4f} | '
              f'f1={f1_mean:.4f}+-{f1_std:.4f}')

    # CHANGE 4: save immediately after each dataset
    save_dataset_result(dataset_id, dataset_result)
    print(f'Dataset {dataset_id} complete in {time.time()-t_ds:.0f}s')

print('\nAll datasets done.')

Dataset  1 — already done, skipping.
Dataset  2 — already done, skipping.
Dataset  3 — already done, skipping.
Dataset  4 — already done, skipping.
Dataset  5 — already done, skipping.
Dataset  6 — already done, skipping.
Dataset  7 — already done, skipping.
Dataset  8 — already done, skipping.
Dataset  9 — already done, skipping.
Dataset 10 — already done, skipping.

=== Dataset 11 | shape=(8330, 265) ===
  fold  1 | SVM_rbf        | start | 3 configs x 5 inner folds
  fold  1 | SVM_rbf        | done  | acc=0.8631 | f1=0.2438 | 47.7s
  fold  1 | SVM_linear     | start | 4 configs x 5 inner folds
  fold  1 | SVM_linear     | done  | acc=0.8571 | f1=0.2247 | 1566.6s
  fold  1 | kNN            | start | 8 configs x 5 inner folds
  fold  1 | kNN            | done  | acc=0.8511 | f1=0.2053 | 6.8s
  fold  1 | DecisionTree   | start | 12 configs x 5 inner folds
  fold  1 | DecisionTree   | done  | acc=0.8631 | f1=0.2496 | 8.2s
  fold  1 | RandomForest   | start | 4 configs x 5 inner folds
  

ValueError: 
All the 20 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/base.py

## 8. Merge All Per-Dataset Files → Single JSON

In [ ]:
baseline_results = {}
missing = []

for dataset_id in range(1, 17):
    data = load_dataset_result(dataset_id)
    if data is None:
        missing.append(dataset_id)
        print(f'WARNING: Dataset {dataset_id} not found')
    else:
        baseline_results[dataset_id] = data

if missing:
    print(f'Missing: {missing}. Re-run Cell 7 to complete them.')
else:
    merged_path = PROJECT_ROOT / 'data' / 'processed' / 'baseline_results.json'
    with merged_path.open('w') as f:
        json.dump(baseline_results, f, indent=2)
    print(f'Merged {len(baseline_results)} datasets → {merged_path}')

## 9. Export to Excel — "Before FS-DR" Sheet

In [ ]:
if missing:
    print('Cannot export — complete all datasets first.')
else:
    results_xlsx = DATASETS_ROOT / 'Results.xlsx'
    wb = load_workbook(results_xlsx)
    ws = wb['Before FS-DR']

    column_map = {
        'SVM':          ('B', 'C', 'L'),
        'kNN':          ('D', 'E', 'M'),
        'DecisionTree': ('F', 'G', 'N'),
        'RandomForest': ('H', 'I', 'O'),
        'MLP':          ('J', 'K', 'P'),
    }

    for dataset_id in range(1, 17):
        start_row = 3 + (dataset_id - 1) * 12
        mean_row  = start_row + 10
        std_row   = start_row + 11

        for clf_name, rows in baseline_results[dataset_id].items():
            acc_col, f1_col, param_col = column_map[clf_name]
            accs, f1s = [], []

            for row in rows:
                excel_row = start_row + row['fold'] - 1
                ws[f'{acc_col}{excel_row}']   = row['accuracy']
                ws[f'{f1_col}{excel_row}']    = row['macro_f1']
                ws[f'{param_col}{excel_row}'] = str(row['best_params'])
                accs.append(row['accuracy'])
                f1s.append(row['macro_f1'])

            acc_mean, acc_std = np.mean(accs), np.std(accs)
            f1_mean,  f1_std  = np.mean(f1s),  np.std(f1s)
            ws[f'{acc_col}{mean_row}'] = f'{acc_mean:.4f} +/- {acc_std:.4f}'
            ws[f'{f1_col}{mean_row}']  = f'{f1_mean:.4f} +/- {f1_std:.4f}'
            ws[f'{acc_col}{std_row}']  = round(acc_std, 4)
            ws[f'{f1_col}{std_row}']   = round(f1_std,  4)

    wb.save(results_xlsx)
    print(f'Exported to: {results_xlsx}')

## 10. Summary Table

In [ ]:
if not baseline_results:
    print('No results yet — run Cells 7 and 8 first.')
else:
    print(f"{'DS':<4} {'Classifier':<14} {'Accuracy':>22} {'Macro-F1':>22}")
    print('-' * 66)
    prev_ds = None
    for dataset_id in sorted(baseline_results.keys(), key=int):
        if prev_ds and prev_ds != dataset_id:
            print()
        for clf_name, rows in baseline_results[dataset_id].items():
            acc = [r['accuracy'] for r in rows]
            f1  = [r['macro_f1'] for r in rows]
            print(f'{dataset_id:<4} {clf_name:<14} '
                  f'{np.mean(acc):.4f} +/- {np.std(acc):.4f}   '
                  f'{np.mean(f1):.4f} +/- {np.std(f1):.4f}')
        prev_ds = dataset_id